# MediBot — RAG Pipeline (LangChain Edition)

This notebook rebuilds the original raw Docling + Qdrant + Groq RAG pipeline
using LangChain abstractions instead. The documents, RBAC access-control logic,
and overall stages stay the same — only the plumbing changes:

| Stage | Before | Now |
|---|---|---|
| Load + chunk | raw `DocumentConverter` + `HybridChunker` | `langchain_docling.DoclingLoader` (same `HybridChunker` underneath) |
| Embeddings | raw `SentenceTransformer` | `langchain_huggingface.HuggingFaceEmbeddings` |
| Vector store | raw `QdrantClient` + manual `upsert()` | `langchain_qdrant.QdrantVectorStore.from_documents()` |
| RBAC filter | Qdrant `Prefetch` filter per vector | Qdrant `Filter` passed into `.as_retriever(search_kwargs=...)` |
| Reranking | raw `CrossEncoder.predict()` + manual sort | `ContextualCompressionRetriever` + `CrossEncoderReranker` |
| Generation | raw `groq.Groq()` call | `langchain_groq.ChatGroq` inside an LCEL chain |

Install the LangChain integration packages first if you haven't already:

```bash
pip install langchain-docling langchain-huggingface langchain-qdrant langchain-groq langchain-core langchain-community
```

In [ ]:
# One-time setup (uncomment if these aren't installed in your environment yet)
# %pip install -q langchain-docling langchain-huggingface langchain-qdrant langchain-groq langchain-core langchain-community

import os
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path(".").resolve()
load_dotenv(ROOT / ".env")

# GROQ_API_KEY must be set before running the generation stage.
# os.environ["GROQ_API_KEY"] = "gsk_..."   # or set it in .env

DEMO_DOC       = str(ROOT / "general" / "staff_handbook.pdf")
COLLECTION     = "mediassist_docs"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL       = "openai/gpt-oss-120b"
QDRANT_PATH     = str(ROOT / "qdrant_lc")   # separate path so it doesn't clash with the raw-client demo
TOP_K           = 10
TOP_K_RERANK    = 3

# Change DEMO_ROLE to test different access levels:
#   doctor | nurse | billing_executive | technician | admin
DEMO_ROLE  = "doctor"
DEMO_QUERY = "What is the leave policy for sick days?"

print("Root      :", ROOT)
print("Demo doc  :", Path(DEMO_DOC).name, "| exists:", Path(DEMO_DOC).exists())
print("Role      :", DEMO_ROLE)

---
## Stage 1 — Load & Chunk

`DoclingLoader` replaces the raw `DocumentConverter` + manual iteration.
We pass it a `HybridChunker` and `export_type=DOC_CHUNKS` so the loader
yields one LangChain `Document` per chunk rather than one giant blob per file.

Each `Document.metadata` carries Docling's heading breadcrumbs under `dl_meta`,
which we surface later for source citations — the same info the raw pipeline
pulled off `chunk.meta.headings`.

In [ ]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

try:
    from docling.chunking import HybridChunker
    chunker = HybridChunker(tokenizer=EMBEDDING_MODEL)
except ImportError:
    from docling_core.transforms.chunker import HierarchicalChunker
    chunker = HierarchicalChunker()

loader = DoclingLoader(
    file_path=DEMO_DOC,
    export_type=ExportType.DOC_CHUNKS,
    chunker=chunker,
)

documents = loader.load()
print(f"Loaded {len(documents)} chunks from: {Path(DEMO_DOC).name}")

---
## Stage 2 — Inspect Chunks

A quick sanity check before embedding: confirm each `Document` has heading
breadcrumbs in `dl_meta` and that the text looks meaningful. This mirrors the
raw pipeline's post-chunk inspection loop, just against LangChain `Document`
objects (`.page_content` / `.metadata`) instead of raw Docling chunk objects.

In [ ]:
def heading_breadcrumb(doc) -> str:
    """Pull a readable heading string from dl_meta, same as the raw pipeline's chunk.meta.headings."""
    dl_meta  = doc.metadata.get("dl_meta", {})
    headings = dl_meta.get("headings") or []
    return " > ".join(headings) if headings else "(no heading)"

for i, doc in enumerate(documents[:3]):
    print(f"--- Chunk {i} ---")
    print("Heading :", heading_breadcrumb(doc))
    print("Text    :", doc.page_content[:200].replace("\n", " ") + "...")
    print()

print(f"Total chunks: {len(documents)}")

---
## Stage 3 — Embed & Index

`HuggingFaceEmbeddings` wraps the same `all-MiniLM-L6-v2` model used in
the raw pipeline. `QdrantVectorStore.from_documents` replaces the manual
`client.create_collection()` + `client.upsert()` calls.

Each stored document keeps its full `.metadata` (including `dl_meta` headings)
alongside the embedding. We also tag every point with `access_roles` — the list
of roles allowed to see this chunk — which the retriever filters on in Stage 4.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# Attach RBAC metadata before indexing so the filter in Stage 4 has something to match on.
# "general" docs are readable by every role; swap this per-document in a full ingestion run.
ACCESS_ROLES = ["doctor", "nurse", "billing_executive", "technician", "admin"]
for doc in documents:
    doc.metadata["access_roles"] = ACCESS_ROLES
    doc.metadata["collection"]   = "general"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vector_store = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    path=QDRANT_PATH,
    collection_name=COLLECTION,
)

print(f"Indexed {len(documents)} chunks into '{COLLECTION}' at '{QDRANT_PATH}'")

---
## Stage 4 — Retrieve (RBAC + Rerank)

Two things happen here that weren't in the raw AtliqAI pipeline:

1. **RBAC filter** — a Qdrant `Filter` passed into `.as_retriever(search_kwargs={"filter": ...})`
   ensures only chunks whose `access_roles` list includes the current user's role
   are returned. Restricted chunks never reach the application.

2. **Cross-encoder reranking** — `ContextualCompressionRetriever` wraps the
   base retriever with a `CrossEncoderReranker`. The cross-encoder scores each
   (query, chunk) pair *jointly* (more accurate than the bi-encoder, but slower),
   then keeps only the top `TOP_K_RERANK` results before they reach the LLM prompt.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchAny
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

# Build the RBAC filter for the current role — mirrors build_qdrant_filter() in rbac.py
rbac_filter = Filter(must=[
    FieldCondition(key="access_roles", match=MatchAny(any=[DEMO_ROLE]))
])

base_retriever = vector_store.as_retriever(
    search_kwargs={"k": TOP_K, "filter": rbac_filter}
)

cross_encoder = HuggingFaceCrossEncoder(model_name=RERANKER_MODEL)
compressor    = CrossEncoderReranker(model=cross_encoder, top_n=TOP_K_RERANK)

retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

# Quick check — inspect what lands after RBAC + reranking
sample_hits = retriever.invoke("What is the leave policy?")
for i, doc in enumerate(sample_hits, start=1):
    print(f"[Source {i}] {heading_breadcrumb(doc)}")
    print(doc.page_content[:150].replace("\n", " ") + "...")
    print()

---
## Stage 5 — Generate

The LCEL chain (`|`) wires everything together:

```
retriever → format_context → prompt → ChatGroq → StrOutputParser
```

`format_context` reproduces the numbered `[Source i]` block the raw pipeline
built manually. The system prompt instructs MediBot to answer **only** from
provided context and to admit when the context is insufficient — no hallucination
fallback. The final `rag()` function mirrors the original `orchestrate()`, returning
both the answer and the source chunks used.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_groq import ChatGroq

SYSTEM_PROMPT = """You are MediBot, the internal knowledge assistant for MediAssist Health Network.
Answer the question using ONLY the context provided below. Be concise and accurate.
If the context is insufficient, state that clearly — do not fabricate information.
Always cite the section name when referencing specific information."""

def format_context(docs) -> str:
    """Same numbered [Source i] block as the raw pipeline's answer_with_context()."""
    return "\n\n".join(
        f"[Source {i} | {heading_breadcrumb(doc)}]\n{doc.page_content}"
        for i, doc in enumerate(docs, start=1)
    )

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0.2,
    api_key=os.environ.get("GROQ_API_KEY"),
)

rag_chain = (
    {"context": retriever | format_context, "question": RunnableLambda(lambda x: x)}
    | prompt
    | llm
    | StrOutputParser()
)


def rag(query: str, role: str = DEMO_ROLE) -> dict:
    """Mirrors orchestrate() in chat.py: retrieves with RBAC, reranks, generates.
    Returns the answer text plus the source chunks used."""
    # Rebuild retriever for the requested role (filter changes per user)
    role_filter = Filter(must=[FieldCondition(key="access_roles", match=MatchAny(any=[role]))])
    role_base   = vector_store.as_retriever(search_kwargs={"k": TOP_K, "filter": role_filter})
    role_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=role_base
    )
    source_docs = role_retriever.invoke(query)
    answer      = (role_retriever | format_context | (lambda ctx: rag_chain.invoke(query))).invoke(query)
    return {
        "answer":  answer,
        "sources": [heading_breadcrumb(d) for d in source_docs],
        "chunks":  source_docs,
    }

In [ ]:
result = rag(DEMO_QUERY, role=DEMO_ROLE)

print(f"Query : {DEMO_QUERY}")
print(f"Role  : {DEMO_ROLE}")
print()
print("ANSWER")
print("──────")
print(result["answer"])
print()
print("Sources used")
print("────────────")
for i, src in enumerate(result["sources"], start=1):
    print(f"  [Source {i}] {src}")